In [ ]:
!pip install -q --upgrade kaggle huggingface_hub
import os
print('--- /kaggle/input contents (diagnostic) ---')
for root, dirs, files in os.walk('/kaggle/input'):
    for name in files:
        print(os.path.join(root, name))


In [ ]:
import os

token_path = '/kaggle/input/hf-token/hf_token.txt'
if not os.path.exists(token_path):
    print('Expected path ' + token_path + ' missing -- searching /kaggle/input for any hf_token*.txt file...')
    found = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for name in files:
            if 'hf_token' in name.lower() or 'hf-token' in name.lower():
                found = os.path.join(root, name)
                break
        if found:
            break
    if not found:
        raise RuntimeError('No hf_token file found anywhere under /kaggle/input.')
    print('Found it at a different path than expected: ' + found)
    token_path = found

with open(token_path) as f:
    os.environ['HF_TOKEN'] = f.read().strip()
print('HF token loaded.')


In [ ]:
import os

cred_path = '/kaggle/input/kaggle-creds/access_token'
if not os.path.exists(cred_path):
    print('Expected path ' + cred_path + ' missing -- searching /kaggle/input for it...')
    found = None
    for root, dirs, files in os.walk('/kaggle/input'):
        for name in files:
            if 'access_token' in name.lower():
                found = os.path.join(root, name)
                break
        if found:
            break
    if not found:
        raise RuntimeError('No Kaggle access_token file found anywhere under /kaggle/input -- the kaggle-creds dataset is not mounted.')
    print('Found it at a different path than expected: ' + found)
    cred_path = found

os.environ['KAGGLE_API_TOKEN'] = cred_path

# Verify auth works before spending 30-40 min downloading models we might not
# be able to upload afterward.
import subprocess
result = subprocess.run(['kaggle', 'datasets', 'list', '-m'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError('Kaggle CLI authentication failed inside the kernel -- see stderr above.')
print('Kaggle CLI authenticated successfully inside the kernel.')


In [ ]:
from huggingface_hub import hf_hub_download
import os

CACHE_DIR = '/kaggle/tmp/model_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

print('Downloading Flux.1-dev unet checkpoint (gated, ~23GB)...')
hf_hub_download(repo_id='black-forest-labs/FLUX.1-dev', filename='flux1-dev.safetensors',
                 local_dir=CACHE_DIR, token=os.environ['HF_TOKEN'])

print('Downloading VAE (gated, small)...')
hf_hub_download(repo_id='black-forest-labs/FLUX.1-dev', filename='ae.safetensors',
                 local_dir=CACHE_DIR, token=os.environ['HF_TOKEN'])

print('Downloading text encoders (ungated)...')
hf_hub_download(repo_id='comfyanonymous/flux_text_encoders', filename='clip_l.safetensors', local_dir=CACHE_DIR)
hf_hub_download(repo_id='comfyanonymous/flux_text_encoders', filename='t5xxl_fp8_e4m3fn.safetensors', local_dir=CACHE_DIR)

print('Downloading USO all-in-one fp8 checkpoint (~16GB, public/ungated)...')
hf_hub_download(repo_id='Comfy-Org/flux1-dev', filename='flux1-dev-fp8.safetensors', local_dir=CACHE_DIR)

print('Downloading USO LoRA (public/ungated)...')
hf_hub_download(repo_id='Comfy-Org/USO_1.0_Repackaged', filename='split_files/loras/uso-flux1-dit-lora-v1.safetensors',
                 local_dir=CACHE_DIR)
# hf_hub_download preserves the repo's internal subfolder structure by
# default -- flatten it so every file in CACHE_DIR sits directly at the
# top level (simplifies both the dataset upload below and the consuming
# notebook's lookup-by-filename logic).
nested = os.path.join(CACHE_DIR, 'split_files', 'loras', 'uso-flux1-dit-lora-v1.safetensors')
flat = os.path.join(CACHE_DIR, 'uso-flux1-dit-lora-v1.safetensors')
if os.path.exists(nested) and not os.path.exists(flat):
    os.rename(nested, flat)
    print('Moved ' + nested + ' -> ' + flat)

print('--- final cache directory contents ---')
!find /kaggle/tmp/model_cache -maxdepth 1 -type f -exec ls -lh {} ;
!df -h /kaggle/tmp


In [ ]:
import json, os, subprocess

CACHE_DIR = '/kaggle/tmp/model_cache'
CACHE_DATASET_ID = 'krkomaleswarreddy/comfyui-flux-uso-model-cache'

metadata = {
    'title': 'ComfyUI Flux USO Model Cache',
    'id': CACHE_DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}
with open(os.path.join(CACHE_DIR, 'dataset-metadata.json'), 'w') as f:
    json.dump(metadata, f)

print('Creating new dataset ' + CACHE_DATASET_ID + '...')
result = subprocess.run(['kaggle', 'datasets', 'create', '-p', CACHE_DIR, '-r', 'skip'],
                         capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

if result.returncode != 0 or 'error' in (result.stdout + result.stderr).lower():
    print('Create failed or dataset may already exist -- trying to create a new version instead...')
    result2 = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', CACHE_DIR, '-m', 'model cache rebuild', '-r', 'skip'],
        capture_output=True, text=True,
    )
    print(result2.stdout)
    print(result2.stderr)
    if result2.returncode != 0:
        raise RuntimeError('Both dataset create and dataset version failed -- see output above.')

print('Done. Check https://www.kaggle.com/datasets/' + CACHE_DATASET_ID)
